In [2]:
!pip install segmentation-models-pytorch albumentations -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.1 MB/s eta 0:00:0000:01


In [1]:
!ls /kaggle/input/datasets/belhadjadji/lastunet17/last_unet_fullimage (3).pth

'last_unet_fullimage (3).pth'


In [3]:
!pip uninstall timm -y
!pip install timm==0.9.2

Found existing installation: timm 1.0.25
Uninstalling timm-1.0.25:
  Successfully uninstalled timm-1.0.25
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 32.2 MB/s eta 0:00:00a 0:00:01


In [4]:
import os, csv, random, cv2, numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm

# ─────────────────────────────────────────────────────────
# STEP 1 — Paths
# ─────────────────────────────────────────────────────────
IMAGES_DIR = "/kaggle/input/datasets/belhadjadji/my-unet-dataset/UNet_dataset_full/images"
MASKS_DIR  = "/kaggle/input/datasets/belhadjadji/my-unet-dataset/UNet_dataset_full/my_masks"

CHECKPOINT_DIR = "/kaggle/working"
BEST_MODEL     = os.path.join(CHECKPOINT_DIR, "best_unet_fullimage.pth")
LAST_MODEL     = os.path.join(CHECKPOINT_DIR, "last_unet_fullimage.pth")
METRICS_CSV    = os.path.join(CHECKPOINT_DIR, "metrics.csv")

RESUME_FROM = None   # ✏️ set to uploaded checkpoint path to resume

# NOTE: Dataset copy to local SSD was removed — /kaggle/working/ does not have
# enough space for 69K images. Reading directly from /kaggle/input/ instead.
# The other speed fixes (AMP, num_workers=4, batch=8) are all still active.

# ─────────────────────────────────────────────────────────
# STEP 2 — Hyperparameters
#
# SPEED CHANGES vs previous version:
#   BATCH_SIZE  : 4  → 8    (fewer kernel launches, better GPU utilization)
#   ACCUM_STEPS : 4  → 2    (effective batch stays 16, half the steps)
#   num_workers : 2  → 4    (uses all Kaggle CPU cores — biggest speedup)
#   persistent_workers: True (workers stay alive between epochs, no restart cost)
#
# Expected: 57 min/epoch → ~10 min/epoch
# If you get CUDA OOM with batch=8: set BATCH_SIZE=6, ACCUM_STEPS=3
# ─────────────────────────────────────────────────────────
IMG_SIZE      = 512
BATCH_SIZE    = 8      # was 4 — more GPU-efficient, same effective batch
ACCUM_STEPS   = 2      # was 4 — effective batch = 8×2 = 16, unchanged
NUM_WORKERS   = 4      # was 2 — uses all Kaggle CPU cores, biggest speedup
EPOCHS        = 40
LR            = 2e-4
WARMUP_EPOCHS = 3
VAL_SPLIT     = 0.15
SEED          = 42
PATIENCE      = 8
EARLY_STOP    = 10

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device      : {device}")
print(f"Batch size  : {BATCH_SIZE} × accum {ACCUM_STEPS} = effective {BATCH_SIZE*ACCUM_STEPS}")
print(f"Num workers : {NUM_WORKERS}")


# ─────────────────────────────────────────────────────────
# STEP 3 — Augmentations
# ─────────────────────────────────────────────────────────
train_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.Rotate(limit=15, p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),    
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])
print("Augmentations defined")


# ─────────────────────────────────────────────────────────
# STEP 4 — Dataset
# ─────────────────────────────────────────────────────────
class FullImageDataset(Dataset):
    def __init__(self, image_paths, masks_dir, transform):
        self.paths = image_paths
        self.mdir  = masks_dir
        self.tf    = transform

        print(f"  Computing mask ratios for {len(image_paths)} images...")
        self.mask_ratios = []
        for p in image_paths:
            stem = os.path.splitext(os.path.basename(p))[0]
            mp   = os.path.join(masks_dir, stem + '.png')
            if os.path.exists(mp):
                m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
                r = float((m > 127).mean()) if m is not None else 0.0
            else:
                r = 0.0
            self.mask_ratios.append(r)

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        img_path  = self.paths[idx]
        stem      = os.path.splitext(os.path.basename(img_path))[0]
        mask_path = os.path.join(self.mdir, stem + '.png')

        image = cv2.imread(img_path)
        if image is None:
            raise FileNotFoundError(f"Cannot read image: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            mask = np.zeros(image.shape[:2], dtype=np.uint8)
        mask = (mask > 127).astype(np.float32)

        aug = self.tf(image=image, mask=mask)
        return aug['image'].float(), aug['mask'].unsqueeze(0).float()


# ─────────────────────────────────────────────────────────
# STEP 5 — Deterministic train/val split
# ─────────────────────────────────────────────────────────
extensions = ('.jpg', '.jpeg', '.png', '.bmp')
all_images  = sorted([
    os.path.join(IMAGES_DIR, f)
    for f in os.listdir(IMAGES_DIR)
    if f.lower().endswith(extensions)
])
assert len(all_images) > 0, f"No images found in {IMAGES_DIR}"

rng = np.random.default_rng(SEED)
shuffled = np.array(all_images)
rng.shuffle(shuffled)
all_images = shuffled.tolist()

n_val       = max(1, int(len(all_images) * VAL_SPLIT))
train_paths = all_images[n_val:]
val_paths   = all_images[:n_val]
print(f"\nDataset split (seed={SEED}):")
print(f"  Train: {len(train_paths)}  Val: {len(val_paths)}")

train_ds = FullImageDataset(train_paths, MASKS_DIR, train_tf)
val_ds   = FullImageDataset(val_paths,   MASKS_DIR, val_tf)


# ─────────────────────────────────────────────────────────
# STEP 6 — WeightedRandomSampler
# ─────────────────────────────────────────────────────────
FLOOR_WEIGHT  = 0.10
sample_weights = [np.sqrt(max(r, FLOOR_WEIGHT)) for r in train_ds.mask_ratios]
mean_w         = np.mean(sample_weights)
sample_weights = [w / mean_w for w in sample_weights]

sampler = WeightedRandomSampler(
    weights     = sample_weights,
    num_samples = len(sample_weights),
    replacement = True,
)

train_loader = DataLoader(
    train_ds,
    batch_size       = BATCH_SIZE,
    sampler          = sampler,
    num_workers      = NUM_WORKERS,   # ← was 2, now 4
    pin_memory       = True,
    drop_last        = True,
    persistent_workers = True,        # ← new: workers stay alive between epochs
)
val_loader = DataLoader(
    val_ds,
    batch_size         = BATCH_SIZE,
    shuffle            = False,
    num_workers        = NUM_WORKERS,
    pin_memory         = True,
    persistent_workers = True,        # ← new
)
print(f"  Batches per epoch: {len(train_loader)}"
      f"  (was {len(train_loader)*2} with batch=4)")


# ─────────────────────────────────────────────────────────
# STEP 7 — Model
# ─────────────────────────────────────────────────────────
model = smp.Unet(
    encoder_name    = "resnet34",
    encoder_weights = "imagenet",
    in_channels     = 3,
    classes         = 1,
    activation      = None,
).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"\nModel parameters: {n_params:,}")


# ─────────────────────────────────────────────────────────
# STEP 8 — Loss: 0.5 BCE + 0.5 Dice
# ─────────────────────────────────────────────────────────
print("\nComputing pos_weight from full training set...")
all_ratios   = [r for r in train_ds.mask_ratios if r > 0]
avg_fg       = float(np.mean(all_ratios)) if all_ratios else 0.05
pos_wt_val   = min((1.0 - avg_fg) / max(avg_fg, 0.01), 10.0)
n_pos_images = sum(1 for r in train_ds.mask_ratios if r > 0.001)
n_neg_images = len(train_ds) - n_pos_images

print(f"  Positive images : {n_pos_images}")
print(f"  Negative images : {n_neg_images}")
print(f"  Mean fg ratio   : {avg_fg:.4f}")
print(f"  BCE pos_weight  : {pos_wt_val:.2f}")

pos_weight = torch.tensor([pos_wt_val], device=device)
bce_fn     = nn.BCEWithLogitsLoss(pos_weight=pos_weight)


def dice_loss(pred, target, smooth=1.0):
    pred   = torch.sigmoid(pred).view(pred.size(0), -1)
    target = target.view(target.size(0), -1)
    inter  = (pred * target).sum(dim=1)
    union  = pred.sum(dim=1) + target.sum(dim=1)
    return 1 - ((2 * inter + smooth) / (union + smooth)).mean()


def loss_fn(pred, target):
    return 0.5 * bce_fn(pred, target) + 0.5 * dice_loss(pred, target)


# ─────────────────────────────────────────────────────────
# STEP 9 — Metrics
# ─────────────────────────────────────────────────────────
def compute_metrics(logits, target, threshold=0.5, eps=1e-7):
    pred   = (torch.sigmoid(logits) > threshold).float()
    pred_f = pred.view(pred.size(0), -1)
    tgt_f  = target.view(target.size(0), -1)
    TP = (pred_f * tgt_f).sum(dim=1)
    FP = (pred_f * (1 - tgt_f)).sum(dim=1)
    FN = ((1 - pred_f) * tgt_f).sum(dim=1)
    TN = ((1 - pred_f) * (1 - tgt_f)).sum(dim=1)
    dice = ((2*TP + eps) / (2*TP + FP + FN + eps)).mean().item()
    iou  = ((TP + eps)   / (TP + FP + FN + eps)).mean().item()
    acc  = ((TP + TN + eps) / (TP + TN + FP + FN + eps)).mean().item()
    prec = ((TP + eps) / (TP + FP + eps)).mean().item()
    rec  = ((TP + eps) / (TP + FN + eps)).mean().item()

    # Background prediction monitor: should decrease toward ~0 over training.
    # If it stays > 0.15, the model still activates too much on background.
    pred_sig = torch.sigmoid(logits)
    bg_mask  = (1 - target)
    bg_sum   = bg_mask.sum().item()
    mean_bg  = ((pred_sig * bg_mask).sum() / bg_sum).item() if bg_sum > 0 else 0.0

    return dice, iou, acc, prec, rec, mean_bg


def compute_metrics_multithreshold(logits, target):
    """Log metrics at 4 thresholds — helps pick best inference threshold."""
    results = {}
    for t in [0.3, 0.4, 0.5, 0.6]:
        d, _, _, p, r, _ = compute_metrics(logits, target, threshold=t)
        f1 = 2*p*r / (p + r + 1e-7)
        results[t] = {'dice': d, 'prec': p, 'rec': r, 'f1': f1}
    return results


# ─────────────────────────────────────────────────────────
# STEP 10 — Optimizer + Scheduler
# ─────────────────────────────────────────────────────────
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=2e-4)
scheduler = ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5,
    patience=PATIENCE, min_lr=1e-6,
)

# ─────────────────────────────────────────────────────────
# SPEED BOOST 2 — Mixed precision (AMP)
#
# Runs forward/backward in FP16, keeps master weights in FP32.
# Halves GPU memory → allows batch_size=16 if you want more speed.
# Typical speedup: 1.5-2x on T4/P100 with no quality loss.
# GradScaler prevents underflow in FP16 gradients.
# ─────────────────────────────────────────────────────────
USE_AMP = torch.cuda.is_available()
scaler  = torch.amp.GradScaler('cuda', enabled=USE_AMP)   # updated API (torch.cuda.amp deprecated)


def atomic_save(obj, path):
    """Write to a temp file first, then rename atomically.
    Prevents partial/corrupt checkpoints if disk fills mid-write,
    and avoids the 'unexpected pos' RuntimeError from torch.save."""
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)   # atomic on Linux — never leaves a corrupt file


def get_warmup_factor(epoch, warmup_epochs):
    if epoch < warmup_epochs:
        return 0.1 + 0.9 * (epoch / warmup_epochs)
    return 1.0


print("Optimizer and scheduler ready")


# ─────────────────────────────────────────────────────────
# STEP 11 — Resume support
# ─────────────────────────────────────────────────────────
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
start_epoch   = 0
best_dice     = 0.0
epochs_no_imp = 0

_ckpt_path = "/kaggle/input/datasets/belhadjadji/lastunet17/last_unet_fullimage (3).pth"
if RESUME_FROM and os.path.exists(RESUME_FROM):
    _ckpt_path = RESUME_FROM
    print(f"\n🔁  Resuming from uploaded checkpoint:\n    {RESUME_FROM}")
elif os.path.exists(LAST_MODEL):
    _ckpt_path = LAST_MODEL
    print(f"\n🔁  Resuming from same-session checkpoint:\n    {LAST_MODEL}")
else:
    print("\n  Starting fresh training")

if _ckpt_path:
    ckpt = torch.load(_ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    start_epoch   = ckpt['epoch'] + 1
    best_dice     = ckpt['best_dice']
    epochs_no_imp = ckpt['epochs_no_imp']
    print(f"  ✅  Epoch {start_epoch} | "
          f"best_dice={best_dice:.4f} | no_imp={epochs_no_imp}/{EARLY_STOP}")

if not os.path.exists(METRICS_CSV):
    with open(METRICS_CSV, 'w', newline='') as f:
        csv.writer(f).writerow([
            'epoch', 'train_loss', 'val_loss',
            'val_dice', 'val_iou', 'val_acc',
            'val_prec', 'val_rec', 'val_mean_bg_pred', 'lr',
            'f1_t0.3', 'f1_t0.4', 'f1_t0.5', 'f1_t0.6',
        ])


# ─────────────────────────────────────────────────────────
# STEP 12 — Training loop
# ─────────────────────────────────────────────────────────
print(f"\nTraining for up to {EPOCHS} epochs "
      f"(early stop after {EARLY_STOP} no-improvement epochs)\n")

for epoch in range(start_epoch, EPOCHS):

    warmup_factor = get_warmup_factor(epoch, WARMUP_EPOCHS)
    for pg in optimizer.param_groups:
        pg['lr'] = LR * warmup_factor

    # ── Train ────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    optimizer.zero_grad()

    loop = tqdm(enumerate(train_loader), total=len(train_loader),
                desc=f'Epoch {epoch+1}/{EPOCHS}', leave=False)
    for step, (images, masks) in loop:
        images, masks = images.to(device), masks.to(device)

        with torch.amp.autocast('cuda', enabled=USE_AMP):
            loss = loss_fn(model(images), masks) / ACCUM_STEPS

        scaler.scale(loss).backward()
        train_loss += loss.item() * ACCUM_STEPS

        if (step + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        loop.set_postfix(loss=f'{train_loss/(step+1):.4f}')

    if len(train_loader) % ACCUM_STEPS != 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

    train_loss /= len(train_loader)

    # ── Validate ─────────────────────────────────────────
    model.eval()
    val_loss = val_dice = val_iou = val_acc = val_prec = val_rec = val_bg = 0.0
    mt_accum = {t: {'dice': 0., 'prec': 0., 'rec': 0., 'f1': 0.}
                for t in [0.3, 0.4, 0.5, 0.6]}

    loop_val = tqdm(val_loader, total=len(val_loader),
                    desc='Validation', leave=False)
    with torch.no_grad():
        for images, masks in loop_val:
            images, masks = images.to(device), masks.to(device)
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                preds = model(images)
            val_loss += loss_fn(preds, masks).item()
            d, i, a, p, r, bg = compute_metrics(preds, masks)
            val_dice += d; val_iou  += i; val_acc  += a
            val_prec += p; val_rec  += r; val_bg   += bg
            mt = compute_metrics_multithreshold(preds, masks)
            for t in mt:
                for k in mt[t]: mt_accum[t][k] += mt[t][k]

    n = len(val_loader)
    val_loss /= n; val_dice /= n; val_iou  /= n
    val_acc  /= n; val_prec /= n; val_rec  /= n; val_bg /= n
    for t in mt_accum:
        for k in mt_accum[t]: mt_accum[t][k] /= n
    current_lr = optimizer.param_groups[0]['lr']

    if epoch >= WARMUP_EPOCHS:
        scheduler.step(val_loss)

    print(f"Epoch {epoch+1:3d}/{EPOCHS} | "
          f"loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
          f"Dice={val_dice:.4f} | IoU={val_iou:.4f} | "
          f"Prec={val_prec:.4f} | Rec={val_rec:.4f} | "
          f"BG_pred={val_bg:.4f} | LR={current_lr:.2e}")
    best_t = max(mt_accum, key=lambda t: mt_accum[t]['f1'])
    print(f"  Threshold scan → best F1 at t={best_t}: "
          + " | ".join(f"t={t} F1={mt_accum[t]['f1']:.3f}"
                       for t in sorted(mt_accum)))

    with open(METRICS_CSV, 'a', newline='') as f:
        csv.writer(f).writerow([
            epoch+1, f"{train_loss:.4f}", f"{val_loss:.4f}",
            f"{val_dice:.4f}", f"{val_iou:.4f}", f"{val_acc:.4f}",
            f"{val_prec:.4f}", f"{val_rec:.4f}", f"{val_bg:.4f}",
            f"{current_lr:.2e}",
            f"{mt_accum[0.3]['f1']:.4f}", f"{mt_accum[0.4]['f1']:.4f}",
            f"{mt_accum[0.5]['f1']:.4f}", f"{mt_accum[0.6]['f1']:.4f}",
        ])

    atomic_save({
        'epoch':         epoch,
        'model':         model.state_dict(),
        'optimizer':     optimizer.state_dict(),
        'scheduler':     scheduler.state_dict(),
        'best_dice':     best_dice,
        'epochs_no_imp': epochs_no_imp,
    }, LAST_MODEL)

    if val_dice > best_dice:
        best_dice     = val_dice
        epochs_no_imp = 0
        atomic_save(model.state_dict(), BEST_MODEL)
        print(f"  ✅  Best model saved  (Dice={best_dice:.4f})")
    else:
        epochs_no_imp += 1
        print(f"  ⏳  No improvement  ({epochs_no_imp}/{EARLY_STOP})")

    if epochs_no_imp >= EARLY_STOP:
        print(f"\n🛑  Early stopping at epoch {epoch+1}")
        break


# ─────────────────────────────────────────────────────────
# STEP 13 — Final summary + zip download
# ─────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print(f"  TRAINING COMPLETE")
print(f"  Best Val Dice  : {best_dice:.4f}")
print(f"  Best model     → {BEST_MODEL}")
print(f"  Last model     → {LAST_MODEL}")
print(f"  Metrics log    → {METRICS_CSV}")
print(f"{'='*55}")

import zipfile
from IPython.display import FileLink

zip_path = "/kaggle/working/training_outputs.zip"
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for file in [BEST_MODEL, LAST_MODEL, METRICS_CSV]:
        if os.path.exists(file):
            zipf.write(file, os.path.basename(file))
            size_mb = os.path.getsize(file) / (1024*1024)
            print(f"  Added: {os.path.basename(file)}  ({size_mb:.1f} MB)")

print(f"\n✅  Zip → {zip_path}")
display(FileLink(zip_path))

Device      : cuda
Batch size  : 8 × accum 2 = effective 16
Num workers : 4
Augmentations defined

Dataset split (seed=42):
  Train: 59501  Val: 10500
  Computing mask ratios for 59501 images...
  Computing mask ratios for 10500 images...
  Batches per epoch: 7437  (was 14874 with batch=4)


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]


Model parameters: 24,436,369

Computing pos_weight from full training set...
  Positive images : 24929
  Negative images : 34572
  Mean fg ratio   : 0.0424
  BCE pos_weight  : 10.00
Optimizer and scheduler ready

  Starting fresh training
  ✅  Epoch 17 | best_dice=0.9475 | no_imp=0/10

Training for up to 40 epochs (early stop after 10 no-improvement epochs)



Epoch  18/40 | loss=0.0418 | val_loss=0.0477 | Dice=0.9489 | IoU=0.9214 | Prec=0.9437 | Rec=0.9672 | BG_pred=0.0027 | LR=2.00e-04
  Threshold scan → best F1 at t=0.6: t=0.3 F1=0.952 | t=0.4 F1=0.954 | t=0.5 F1=0.955 | t=0.6 F1=0.956
  ✅  Best model saved  (Dice=0.9489)


...

Epoch  37/40 | loss=0.0331 | val_loss=0.0404 | Dice=0.9545 | IoU=0.9293 | Prec=0.9465 | Rec=0.9704 | BG_pred=0.0024 | LR=2.00e-04
  Threshold scan → best F1 at t=0.6: t=0.3 F1=0.956 | t=0.4 F1=0.957 | t=0.5 F1=0.958 | t=0.6 F1=0.959
  ⏳  No improvement  (7/10)


Epoch  38/40 | loss=0.0331 | val_loss=0.0408 | Dice=0.9545 | IoU=0.9291 | Prec=0.9460 | Rec=0.9702 | BG_pred=0.0023 | LR=2.00e-04
  Threshold scan → best F1 at t=0.6: t=0.3 F1=0.956 | t=0.4 F1=0.957 | t=0.5 F1=0.958 | t=0.6 F1=0.958
  ⏳  No improvement  (8/10)


Epoch  39/40 | loss=0.0322 | val_loss=0.0420 | Dice=0.9535 | IoU=0.9281 | Prec=0.9460 | Rec=0.9699 | BG_pred=0.0023 | LR=2.00e-04
  Threshold scan → best F1 at t=0.6: t=0.3 F1=0.955 | t=0.4 F1=0.957 | t=0.5 F1=0.957 | t=0.6 F1=0.958
  ⏳  No improvement  (9/10)


Epoch  40/40 | loss=0.0324 | val_loss=0.0411 | Dice=0.9534 | IoU=0.9281 | Prec=0.9461 | Rec=0.9686 | BG_pred=0.0024 | LR=2.00e-04
  Threshold scan → best F1 at t=0.6: t=0.3 F1=0.955 | t=0.4 F1=0.956 | t=0.5 F1=0.957 | t=0.6 F1=0.958
  ⏳  No improvement  (10/10)

🛑  Early stopping at epoch 40

  TRAINING COMPLETE
  Best Val Dice  : 0.9547
  Best model     → /kaggle/working/best_unet_fullimage.pth
  Last model     → /kaggle/working/last_unet_fullimage.pth
  Metrics log    → /kaggle/working/metrics.csv
  Added: best_unet_fullimage.pth  (93.4 MB)
  Added: last_unet_fullimage.pth  (279.9 MB)
  Added: metrics.csv  (0.0 MB)

✅  Zip → /kaggle/working/training_outputs.zip


/kaggle/working/training_outputs.zip